# Fase 1: Módulo Cuantitativo y Entrenamiento MLP / LSTM en Google Colab GPU T4

**Arquitectura Híbrida del Proyecto:**
- **Modelo Cuantitativo Base (Front Office):** MLP y LSTM entrenados en GPU NVIDIA T4 con escalador ajustado **exclusivamente** en `TR`.
- **Dual Laya (`laya:latest`):** Ejecutado en dispositivo local vía Ollama `/v1/systemone`.
- **Orquestador Qwen 14B (`qwen2.5-14b-instruct`):** Consumido por **API Externa Remota** (no corre en local).

### Universo Oficial de 30 Tickers
`['AAPL', 'MSFT', 'NVDA', 'TSLA', 'HWM', 'JPM', 'GOOGL', 'AMZN', 'META', 'LMT', 'WM', 'LULU', 'DIS', 'V', 'MA', 'BTC-USD', 'SPY', 'QQQ', 'NFLX', 'AMD', 'CRM', 'ADBE', 'PYPL', 'COST', 'HD', 'UNH', 'JNJ', 'PG', 'KO', 'WMT']`

In [ ]:
# 1. Verificación de Hardware (GPU NVIDIA T4 en Colab) y Versiones
import subprocess
import torch
import sklearn
import pandas as pd
import numpy as np

print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
    try:
        print(subprocess.check_output(['nvidia-smi'], text=True))
    except Exception as e:
        print('nvidia-smi no disponible:', e)
else:
    print('Ejecutando en modo CPU reproducible (en Colab activar Runtime -> T4 GPU).')

In [ ]:
# 2. Definición del Universo de 30 Activos y Contrato de Features (Sin Data Leakage)
PRIMARY_TICKERS = [
    'AAPL', 'MSFT', 'NVDA', 'TSLA', 'HWM', 'JPM', 'GOOGL', 'AMZN', 'META', 'LMT',
    'WM', 'LULU', 'DIS', 'V', 'MA', 'BTC-USD', 'SPY', 'QQQ', 'NFLX', 'AMD',
    'CRM', 'ADBE', 'PYPL', 'COST', 'HD', 'UNH', 'JNJ', 'PG', 'KO', 'WMT'
]

FEATURE_COLUMNS = [
    'ret_lag_1', 'ret_lag_2', 'ret_lag_3', 'ret_lag_5',
    'ema_ratio', 'vol_20', 'rsi_14', 'momentum_5'
]
print(f'Total tickers configurados: {len(PRIMARY_TICKERS)} | Features: {FEATURE_COLUMNS}')

In [ ]:
# 3. Ejecución Completa de Descarga, Partición Temporal (TR 70% / VA 15% / TE 15%),
# Entrenamiento MLP/LSTM (3 configs x 3 semillas, 3 ventanas expansivas, ablación sin indicadores),
# Evaluación en TE (MAE, RMSE, Directional Accuracy, Win Rate, Max Drawdown) y Exportación de Artefactos.
from pathlib import Path
import sys
repo_root = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from notebooks.generate_phase1_artifacts import run_phase1
run_phase1()

## 4. Documentación de los 4 Fallos Deliberados de la Práctica

1. **Fallo Deliberado #1 — Data Leakage al ajustar `StandardScaler` sobre `TR+VA+TE`:**
   * *Causa:* Ajustar `scaler.fit(full_df)` contamina la media y varianza con precios futuros.
   * *Síntoma:* Mejora artificial del RMSE en `TE` que colapsa en producción real.
   * *Corrección:* `scaler.fit()` se ejecuta **estrictamente sobre `TR`**; `VA` y `TE` solo llaman `scaler.transform()`.

2. **Fallo Deliberado #2 — Shuffle aleatorio en lugar de partición temporal (`train_test_split(..., shuffle=True)`):**
   * *Causa:* Mezclar ventanas temporales solapadas permite que el modelo memorice días adyacentes futuros.
   * *Corrección:* Partición cronológica estricta `TR` (70% inicial) $\rightarrow$ `VA` (15% intermedio) $\rightarrow$ `TE` (15% final) imprimiendo fechas exactas en `model_manifest.json`.

3. **Fallo Deliberado #3 — Uso de vela abierta / sesgo de anticipación ($t$ ejecutado al cierre $t$):**
   * *Causa:* Usar el precio de cierre de la vela $t$ cuando la decisión se toma con indicadores calculados al cierre de $t$ (o usar `bfill()`).
   * *Corrección:* Descarte estricto de vela abierta (`close_time < current_utc_time`), eliminación de `.bfill()`, y ejecución simulada en $t+1$.

4. **Fallo Deliberado #4 — División entre cero en Precisión Direccional y Win Rate sin ciclos cerrados:**
   * *Causa:* Calcular `hits / N` con rendimientos reales iguales a `0` o `wins / closed_trades` cuando no se completó ningún ciclo Compra+Venta.
   * *Corrección:* Retornar explícitamente `"No definido"` cuando el denominador es `0` y excluir `y_i == 0` de la precisión direccional.

In [ ]:
# 5. Inspección de Artefactos Generados y CSVs Exportados
import json
manifest = json.loads((repo_root / 'notebooks' / 'artifacts' / 'model_manifest.json').read_text(encoding='utf-8'))
print('=== MANIFEST DEL MODELO CONGELADO ===')
print(json.dumps(manifest['splits_summary'], indent=2))
print(json.dumps(manifest['metrics_TE'], indent=2))

exp_df = pd.read_csv(repo_root / 'exports' / 'experimentos.csv')
display(exp_df)

fut_df = pd.read_csv(repo_root / 'exports' / 'prediccion_futura.csv')
print('Predicciones futuras (30 tickers, columna real vacía):')
display(fut_df.head(10))